# DeFoG — Marginal Ablation on Planar (Colab)

Three distributions trained from scratch, ~9h total on A100. Mirrors the QM9
ablation but with non-molecular structural-graph generation: validity here
means "the generated graph is planar."

| Run | Transition | Marginals source |
|-----|-----------|------------------|
| `planar_ablation_uniform`  | `uniform`         | — |
| `planar_ablation_marginal` | `marginal`        | dataset type-frequencies |
| `planar_ablation_spminer`  | `loaded_marginal` | SPMiner motifs mined from Planar training graphs |

### Configuration (Config A)
- 5,000 epochs per run (10K training steps; ~10% of paper figure)
- 20 checkpoints per run (validate every 250 epochs)
- 32 mid-training samples × 200 sample_steps (per-validation cost: ~8 min)
- Final test: 128 samples × 1000 sample_steps (paper-grade per-run number)

### Prerequisites
- GPU runtime (A100 strongly preferred; T4 is ~3× slower).
- `spminer_motif_marginals.pt` is in `MyDrive/DeFoGColab/data/planar/`
  (created by Phases 1 + 2 of the SPMiner pipeline before this notebook).
- `WANDB_API_KEY` saved as a Colab Secret.

### Expected runtime
~3h per run × 3 = ~9h total (well within a 12h Colab session).

## 1 · Environment setup

Identical to the QM9 ablation notebook — `colab_create_env.sh` builds the
defog conda env from scratch each session, with all known dependency-conflict
workarounds baked in.

In [ ]:
# Mount Drive and load W&B API key from Colab Secrets.
from google.colab import drive, userdata
import os

drive.mount('/content/drive')

try:
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    print('W&B API key loaded from Colab Secrets.')
except Exception as e:
    print(f'Warning: could not load WANDB_API_KEY ({e}).')
    print('Set manually if needed: os.environ["WANDB_API_KEY"] = "<your-key>"')

os.environ['MPLBACKEND'] = 'Agg'
print('Drive mounted.')

In [ ]:
# Verify GPU. CUDA version determines which PyTorch wheel will be used.
!nvidia-smi

In [ ]:
%%bash
# Install Miniforge if absent. ~30s on first install, instant on re-runs.
set -e
if [ -f /content/miniconda3/bin/conda ]; then
    echo "Conda already installed."
    /content/miniconda3/bin/conda --version
    exit 0
fi
wget -q \
  https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh \
  -O /tmp/miniforge.sh
bash /tmp/miniforge.sh -b -p /content/miniconda3
rm /tmp/miniforge.sh
echo "Miniforge installed."
/content/miniconda3/bin/conda --version

In [ ]:
# Set this to your fork's URL.
GITHUB_REPO = 'https://github.com/Maxlyu254/DeFoGPlus.git'
REPO_DIR    = '/content/DeFoGPlus'

print(f'Will clone: {GITHUB_REPO}')
print(f'       to : {REPO_DIR}')

In [ ]:
%%bash -s "$GITHUB_REPO" "$REPO_DIR"
GITHUB_REPO="$1"
REPO_DIR="$2"
set -e
if [ -d "$REPO_DIR/.git" ]; then
    echo "Repo already present — pulling latest."
    git -C "$REPO_DIR" pull
else
    git clone "$GITHUB_REPO" "$REPO_DIR"
fi
echo "Repo ready."
git -C "$REPO_DIR" log --oneline -3

In [ ]:
%%bash
# Build the defog conda env from scratch (~8-12 min).
# Always rebuilds: deletes any existing /content/miniconda3/envs/defog first.
set -e
bash /content/DeFoGPlus/shell/colab_create_env.sh 2>&1

In [ ]:
%%bash -s "$REPO_DIR"
# Wire Drive paths via symlinks so checkpoints persist across sessions.
REPO_DIR="$1"
set -e

DRIVE_BASE='/content/drive/MyDrive/DeFoGColab'
mkdir -p "$DRIVE_BASE/checkpoints" "$DRIVE_BASE/data/planar"

CKPT_LINK="$REPO_DIR/src/checkpoints"
[ -L "$CKPT_LINK" ] && rm "$CKPT_LINK"
ln -s "$DRIVE_BASE/checkpoints" "$CKPT_LINK"
echo "Symlink: $CKPT_LINK -> $DRIVE_BASE/checkpoints"

DATA_LINK="$REPO_DIR/data"
[ -L "$DATA_LINK" ] || [ -d "$DATA_LINK" ] && rm -rf "$DATA_LINK"
ln -s "$DRIVE_BASE/data" "$DATA_LINK"
echo "Symlink: $DATA_LINK -> $DRIVE_BASE/data"

echo "Symlinks ready."

In [ ]:
%%bash
# Compile the ORCA orbit-counting binary used for non-molecular graph metrics
# (MMD evaluation on Planar/SBM). The source ships with the repo but the
# binary must be built locally. QM9 doesn't need this because it uses rdkit
# metrics instead.
set -e
ORCA_DIR=/content/DeFoGPlus/src/analysis/orca
if [ -x "$ORCA_DIR/orca" ]; then
    echo "ORCA binary already compiled."
else
    echo "Compiling ORCA..."
    cd "$ORCA_DIR"
    g++ -O2 -std=c++11 -o orca orca.cpp
    echo "ORCA compiled: $(ls -la orca)"
fi

In [ ]:
# Define BASE_ENV once — every subprocess inherits these.
#
# LD_PRELOAD: forces conda's libgomp to load before torch's bundled one,
#   preventing the 'GOMP_5.0 not found' error when graph_tool imports.
# MPLBACKEND: overrides Jupyter's inline backend (not present in conda env).
# PYTHONPATH: lets `from src import utils` resolve when running main.py from
#   inside src/ (sys.path[0] otherwise points to src/, not REPO_DIR).
import subprocess, os

PYTHON      = '/content/miniconda3/envs/defog/bin/python'
DRIVE_BASE  = '/content/drive/MyDrive/DeFoGColab'
LIBGOMP     = '/content/miniconda3/envs/defog/lib/libgomp.so.1'
SRC         = f'{REPO_DIR}/src'
DATA_PLANAR = f'{REPO_DIR}/data/planar'

BASE_ENV = {
    **os.environ,
    'MPLBACKEND': 'Agg',
    'LD_PRELOAD': LIBGOMP,
    'PYTHONPATH': REPO_DIR,
}

# Verify imports work end-to-end with the full BASE_ENV.
check = '''
import torch, graph_tool, rdkit, torch_geometric, pytorch_lightning, hydra, wandb, imageio
import numpy as np
assert hasattr(np, "Inf"), "np.Inf missing — pytorch-lightning will crash"
print("torch            :", torch.__version__)
print("graph_tool       :", graph_tool.__version__)
print("numpy            :", np.__version__)
print("CUDA available   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU              :", torch.cuda.get_device_name(0))
'''
r = subprocess.run([PYTHON, '-c', check], capture_output=True, text=True, env=BASE_ENV)
print(r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr)
    raise RuntimeError('Import check failed.')

# W&B auth
r2 = subprocess.run([PYTHON, '-m', 'wandb', 'whoami'],
                    capture_output=True, text=True, env=BASE_ENV)
print('W&B identity:', r2.stdout.strip() or '(not logged in)')
print('\nEnvironment ready.')

## 2 · Configuration

Per-distribution Hydra overrides and Config A training/validation settings.

In [ ]:
# All three runs start from scratch (no prior Planar checkpoints).
SPMINER_PT = f'{DATA_PLANAR}/spminer_motif_marginals.pt'

NAMES = {
    'uniform':  'planar_ablation_uniform',
    'marginal': 'planar_ablation_marginal',
    'spminer':  'planar_ablation_spminer',
}

EXTRAS = {
    'uniform':  ['model.transition=uniform'],
    'marginal': ['model.transition=marginal'],
    'spminer':  ['model.transition=loaded_marginal',
                 f'model.motif_marginals_path={SPMINER_PT}'],
}

# ── TRAIN config ──────────────────────────────────────────────────────────────
# Config A from the design discussion:
#   train.n_epochs=5000              -> 5000 × 2 steps_per_epoch = 10K steps (~10% of paper figure)
#   check_val_every_n_epochs=250     -> 20 validations / 20 checkpoints
#   sample_every_val=1               -> save a checkpoint at every validation
#   samples_to_generate=32           -> mid-training samples (±9% noise on % planar)
#   sample.sample_steps=200          -> 5× faster validation than paper's 1000
#
# IMPORTANT: `dataset=planar` is REQUIRED. `+experiment=planar` only overrides
# scalar values — it does NOT switch the dataset config group. Without
# `dataset=planar`, the default qm9 dataset is loaded and you'd train a
# Planar-configured model on QM9 data (atoms, bonds, batch_size=64 → 1500
# steps/epoch). Same bug as in run_spminer.sh.
TRAIN_COMMON = [
    '+experiment=planar',
    'dataset=planar',
    'hydra.job.chdir=False',
    'train.n_epochs=5000',
    'general.check_val_every_n_epochs=250',
    'general.sample_every_val=1',
    'general.samples_to_generate=32',
    'sample.sample_steps=200',
]

# ── TEST config ───────────────────────────────────────────────────────────────
# Final-model test uses paper-grade settings for a clean per-run number:
#   final_model_samples_to_generate=128   -> ±2-3% noise floor on % planar
#   sample.sample_steps=1000              -> paper's full denoising budget
TEST_COMMON = [
    '+experiment=planar',
    'dataset=planar',
    'hydra.job.chdir=False',
    'general.final_model_samples_to_generate=128',
    'sample.sample_steps=1000',
]

print('Configuration loaded.')
for k, v in NAMES.items():
    print(f'  {k:9s} → {v}')

In [ ]:
# Helpers: subprocess streamer, checkpoint discovery, train_run, test_run.
import glob


def find_latest_checkpoint(name):
    """Return the most recent .ckpt path under src/checkpoints/{name}/, or None."""
    ckpt_dir = f'{SRC}/checkpoints/{name}'
    ckpts = sorted(glob.glob(f'{ckpt_dir}/*.ckpt'), key=os.path.getmtime)
    return ckpts[-1] if ckpts else None


def run_live(cmd, cwd=SRC):
    """Stream subprocess stdout+stderr line by line. Raise on non-zero exit."""
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, cwd=cwd, env=BASE_ENV,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'Process exited with code {proc.returncode}')


def train_run(key):
    """Resume-aware training for one distribution.

    If a checkpoint exists in src/checkpoints/{name}/, append
    general.resume='<latest>' so Lightning continues from that epoch.
    Otherwise start from scratch. n_epochs=5000 is the stop-at-epoch target.
    """
    name   = NAMES[key]
    extras = EXTRAS[key]
    args = TRAIN_COMMON + [f'general.name={name}'] + extras

    latest = find_latest_checkpoint(name)
    if latest:
        # Single-quote: Lightning's checkpoint filenames contain '=' which
        # collides with Hydra's key=value override syntax otherwise.
        args.append(f"general.resume='{latest}'")
        print(f'Resuming {name} from: {latest}')
    else:
        print(f'Starting {name} from scratch.')

    print(f'\n{"╔" + "═"*58 + "╗"}')
    print(f'  TRAIN  →  {name}')
    print(f'{"╚" + "═"*58 + "╝"}\n')
    run_live([PYTHON, 'main.py'] + args)
    print(f'\n✓  {name} training complete.')


def test_run(key):
    """Run final-model test on the latest checkpoint of one distribution.

    Uses paper-grade sampling: 128 samples × 1000 sample_steps for a
    low-noise final number. Cost: ~3-5 min per test cell.
    """
    name   = NAMES[key]
    extras = EXTRAS[key]

    latest = find_latest_checkpoint(name)
    if not latest:
        raise RuntimeError(f'No checkpoint found for {name}. Train first.')

    args = (
        TEST_COMMON
        + [f'general.name={name}']
        + extras
        + [f"general.test_only='{latest}'"]
    )

    print(f'\n{"╔" + "═"*58 + "╗"}')
    print(f'  TEST   →  {name}')
    print(f'{"╚" + "═"*58 + "╝"}')
    print(f'Checkpoint: {latest}\n')
    run_live([PYTHON, 'main.py'] + args)
    print(f'\n✓  {name} test complete.')


print('Helpers loaded.')

## 3 · Pre-flight checks

Verify GPU, marginals file, and report which checkpoints will be resumed
(none expected on a first run).

In [ ]:
import torch

errors = []

# GPU
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    print(f'✓  GPU: {gpu}')
    if 'A100' not in gpu:
        print(f'   ⚠  Not an A100 — time estimates assume A100, expect ~3× slower on T4.')
else:
    errors.append('No CUDA GPU detected — training will be unusably slow.')

# SPMiner marginals .pt file (required for the spminer run)
spminer_pt = f'{DATA_PLANAR}/spminer_motif_marginals.pt'
if os.path.isfile(spminer_pt):
    m = torch.load(spminer_pt, map_location='cpu', weights_only=True)
    print(f'✓  Planar SPMiner marginals: X={list(m["X"].shape)}  E={list(m["E"].shape)}')
else:
    errors.append(
        f'Missing: {spminer_pt}\n'
        f'   Run Phases 1+2 of the SPMiner pipeline on Planar first '
        f'(see project README).'
    )

# Symlinks
for link, target in [
    (f'{REPO_DIR}/src/checkpoints', f'{DRIVE_BASE}/checkpoints'),
    (f'{REPO_DIR}/data',            f'{DRIVE_BASE}/data'),
]:
    ok = os.path.islink(link) and os.readlink(link) == target
    print(f'  {"✓" if ok else "✗"} symlink  {link}')

# Existing checkpoints — show what will be resumed (likely none)
print('\nResume status:')
for key, name in NAMES.items():
    latest = find_latest_checkpoint(name)
    if latest:
        print(f'  {key:9s} → resume from {os.path.basename(latest)}')
    else:
        print(f'  {key:9s} → fresh start')

# W&B
if os.environ.get('WANDB_API_KEY', ''):
    print('\n✓  WANDB_API_KEY is set.')
else:
    print('\n⚠  WANDB_API_KEY not set — runs will not be logged to W&B.')

if errors:
    print()
    for e in errors:
        print(f'✗  {e}')
    raise RuntimeError('Pre-flight failed.')
print('\nReady to train/test.')

## 4 · Training

Three independent training cells, each resume-aware. Run them sequentially —
all three together fit in ~9h on A100.

If a session times out mid-run, re-running the cell next session continues
from the latest saved checkpoint (worst-case loss: up to 250 epochs of work).

In [ ]:
# Train uniform — fresh start, ~3h on A100.
# (5000 epochs × ~2.3 sec training + 20 validations × ~8 min sampling)
train_run('uniform')

In [ ]:
# Train marginal — fresh start, ~3h on A100.
train_run('marginal')

In [ ]:
# Train spminer — fresh start, ~3h on A100.
train_run('spminer')

## 5 · Testing

Each test cell auto-picks the latest checkpoint of its run and generates 128
molecules with paper-grade sampling (sample_steps=1000) for a clean per-run
final number. ~3-5 min per cell.

In [ ]:
test_run('uniform')

In [ ]:
test_run('marginal')

In [ ]:
test_run('spminer')

## 6 · Summary

In [ ]:
print('=' * 60)
print('  Planar ablation summary')
print('=' * 60)

for key, name in NAMES.items():
    print(f'\n{name}')
    ckpt_dir = f'{SRC}/checkpoints/{name}'
    ckpts = sorted(glob.glob(f'{ckpt_dir}/*.ckpt'), key=os.path.getmtime)
    print(f'  Checkpoints saved ({len(ckpts)}):')
    for c in ckpts:
        print(f'    {os.path.basename(c)}')

print()
print('View training curves in W&B:')
print('  - Set X-axis to "epoch" for matching across runs.')
print('  - Validity metric on Planar = fraction of generated graphs that are planar.')
print('  - Compare uniform / marginal / spminer side-by-side in a chart panel.')
print()
print(f'Checkpoints persisted at:\n  {DRIVE_BASE}/checkpoints/')